# 3 - Customer Spending Prediction - Regression

<img src='https://camoinassociates.com/wp-content/uploads/2024/04/Consumer-Spending.jpg'>


Bu çalışmada müşteri verilerini kullanarak harcama miktarını tahmin eden bir regression modeli geliştireceğiz.

## Akış

1. Veriyi yükleme
2. Veriyi inceleme
3. Veri temizleme
4. Feature engineering
5. Kategorik verileri sayısala çevirme
6. Regression modelleri kurma
7. Model sonuçlarını karşılaştırma
8. Sonuç


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Veriyi Yükleme


In [2]:
# Bu projede Kaggle'dan indirilen dataseti Colab üzerinden zip dosyası olarak açıp kullanacağım.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
import zipfile

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/Customer_Spending_Dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')


In [7]:
import os
os.listdir('/content')[:50]


['.config', 'customer_data.csv', 'drive', 'sample_data']

## 2. Veriyi Okuma ve İnceleme


In [8]:
# Bu bölümde zip içinden çıkan gerçek csv dosyasını okuyup veri setinin yapısını inceleyeceğim.


In [9]:
file_path = '/content/customer_data.csv'

df = pd.read_csv(file_path)
df.head()


,name,age,gender,education,income,country,purchase_frequency,spending
0,Teresa Williams MD,42,Female,High School,53936,Slovenia,0.9,13227.120
1,Christine Myers,49,Female,Master,82468,Aruba,0.6,12674.040
2,Dwayne Moreno,55,Male,Bachelor,56941,Cyprus,0.3,5354.115
3,Amy Norton,24,Female,Bachelor,60651,Palau,0.2,2606.510
4,Tonya Adams,64,Male,Master,81884,Zambia,0.9,18984.780


In [10]:
df.shape


(1000, 8)

In [11]:
df.columns


Index(['name', 'age', 'gender', 'education', 'income', 'country',
       'purchase_frequency', 'spending'],
      dtype='object')

In [12]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                1000 non-null   object 
 1   age                 1000 non-null   int64  
 2   gender              1000 non-null   object 
 3   education           1000 non-null   object 
 4   income              1000 non-null   int64  
 5   country             1000 non-null   object 
 6   purchase_frequency  1000 non-null   float64
 7   spending            1000 non-null   float64
dtypes: float64(2), int64(2), object(4)
memory usage: 62.6+ KB


## 3. Veri Temizleme


In [13]:
# Bu bölümde tekrar eden satırları, boş verileri ve veri tiplerini kontrol edeceğim.


In [14]:
df.duplicated().sum()


np.int64(0)

In [15]:
df = df.drop_duplicates()


In [16]:
df.isnull().sum()


,0
name,0
age,0
gender,0
education,0
income,0
country,0
purchase_frequency,0
spending,0


## 4. Feature Engineering


In [18]:
df = df.drop('name', axis=1)
df.head()


,age,gender,education,income,country,purchase_frequency,spending
0,42,Female,High School,53936,Slovenia,0.9,13227.120
1,49,Female,Master,82468,Aruba,0.6,12674.040
2,55,Male,Bachelor,56941,Cyprus,0.3,5354.115
3,24,Female,Bachelor,60651,Palau,0.2,2606.510
4,64,Male,Master,81884,Zambia,0.9,18984.780


In [20]:
# Herhangi bir işlem yapmaya gerek yok.


## 5. Kategorik Verileri Sayısala Çevirme


In [21]:
# Model kurmadan önce kategorik sütunları sayısal hale getireceğim.


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   age                 1000 non-null   int64  
 1   gender              1000 non-null   object 
 2   education           1000 non-null   object 
 3   income              1000 non-null   int64  
 4   country             1000 non-null   object 
 5   purchase_frequency  1000 non-null   float64
 6   spending            1000 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 54.8+ KB


In [24]:
df = pd.get_dummies(df, drop_first=True)
df.head()


,age,income,purchase_frequency,spending,gender_Male,education_High School,education_Master,education_PhD,country_Albania,country_Algeria,...,country_Uruguay,country_Uzbekistan,country_Vanuatu,country_Venezuela,country_Vietnam,country_Wallis and Futuna,country_Western Sahara,country_Yemen,country_Zambia,country_Zimbabwe
0,42,53936,0.9,13227.120,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,49,82468,0.6,12674.040,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,55,56941,0.3,5354.115,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,24,60651,0.2,2606.510,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,64,81884,0.9,18984.780,True,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [25]:
df.shape

(1000, 246)

## 6. Regression Modelleri


In [26]:
# Bu bölümde birkaç farklı regression modeli kurup sonuçları karşılaştıracağım.


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

x = df.drop('spending', axis=1)
y = df['spending']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [28]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append([name, mse, r2])

results_df = pd.DataFrame(results, columns=['Model', 'MSE', 'R2'])
results_df


,Model,MSE,R2
0,Linear Regression,2.079322e+06,0.931137
1,Ridge Regression,1.739190e+06,0.942402
2,Lasso Regression,1.819943e+06,0.939727
3,Random Forest,7.416819e+05,0.975437
4,Gradient Boosting,2.195136e+05,0.992730
5,Extra Trees,8.255440e+05,0.972660


## 7. Sonuçları Karşılaştırma


In [29]:
# Burada hangi modelin daha başarılı olduğuna bakacağım.


In [30]:
results_df.sort_values('R2', ascending=False)


,Model,MSE,R2
4,Gradient Boosting,2.195136e+05,0.992730
3,Random Forest,7.416819e+05,0.975437
5,Extra Trees,8.255440e+05,0.972660
1,Ridge Regression,1.739190e+06,0.942402
2,Lasso Regression,1.819943e+06,0.939727
0,Linear Regression,2.079322e+06,0.931137


## Sonuç


Bu çalışmada müşteri verileri kullanılarak harcama miktarını tahmin eden regression modelleri denendi. Elde edilen sonuçlara göre en başarılı model Gradient Boosting oldu ve model 0.9927 R2 skoru elde etti.
